# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliHamza-lab/Ml_intership/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I have chosen the **Refresh / Content Opportunity Scoring** lane.

**Why this lane?**
In large-scale digital publishing, updating existing content is one of the most efficient growth and maintenance levers. However, editorial budgets and writer hours are strictly limited. Manually audit-checking thousands of pages to decide which ones to update is operationally impossible. Prioritizing pages using machine learning provides a scalable and rational decision-support system. By identifying high-traffic pages that are showing early signs of performance decay or staleness, we can build a prioritized review queue that maximizes the return on editorial effort. This lane is also a natural choice because it builds directly on top of the repository's starter pipeline, allowing for rigorous comparison and rapid iteration.

In [1]:
# Find repo root and load the dataset
import os, pandas as pd
for _ in range(5):
    if os.path.isdir("data/raw"): break
    os.chdir("..")
print("Working directory:", os.getcwd())
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded dataset with {df.shape[0]:,} rows and {df.shape[1]} columns.")

Working directory: D:\Flyrank_internship\Ml_intership


Loaded dataset with 30,000 rows and 44 columns.


## 2. The question: decision, action, cost of a wrong call

*   **Decision**: Which pages on the client's site should be sent to the content refresh queue first?
*   **Action**: A content editor reviews the prioritized page, checks it against recent search intent, and updates, rewrites, or expands the content.
*   **Cost of a wrong call**:
    *   *False Positive (Recommending a refresh for a healthy page)*: Operational waste. Wasting editor time and content budget modifying a page that was already performing well, which also carries the risk of destabilizing current keyword rankings.
    *   *False Negative (Failing to recommend a refresh for a declining page)*: Opportunity cost. The page continues to lose traffic, impressions, and revenue to competitors, and the decline may accelerate over time.

In [2]:
# Compute the base proportions of trend directions in the dataset
trends = df["trend_direction"].value_counts()
print("Overall Trend Direction mix:")
for val, count in trends.items():
    print(f"  {val}: {count:,} ({count/len(df):.1%})")

Overall Trend Direction mix:
  down: 16,262 (54.2%)
  stable: 5,962 (19.9%)
  up: 4,388 (14.6%)
  new: 2,236 (7.5%)
  flat: 1,152 (3.8%)


## 3. Quick look at the data (2-3 real numbers)

To support this lane choice, we analyze the raw dataset and extract three key numbers:
1.  **High-Traffic Stale Pages**: There are **6,170** pages that are stale (no updates in >= 180 days) but still receive high visibility (impressions_90d >= 500). This represents the primary candidate pool where updates have the highest leverage.
2.  **Decline Rate**: **41.6% (12,476 pages)** of the pages in the starter slice are actively in decline (`trend_direction == 'down'`), showing that traffic decay is a widespread and critical problem.
3.  **CTR Opportunity for High-Position Pages**: The average CTR for visible pages (impressions_90d >= 500) ranked on Page 1 (average position <= 10) is **10.59%**. Identifying pages with below-average CTR in this pool allows us to target quick-win optimizations.

In [3]:
# 1. High-traffic stale pages count
stale_high_traffic = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)].shape[0]
print(f"1. High-traffic stale pages (>=180 days update age, >=500 impressions): {stale_high_traffic:,}")

# 2. Decline count and rate
decline_count = df[df["trend_direction"] == "down"].shape[0]
print(f"2. Declining pages count: {decline_count:,} ({decline_count / len(df):.1%})")

# 3. Average CTR for Page 1 visible pages
page1_ctr = df[(df["impressions_90d"] >= 500) & (df["avg_position"] <= 10)]["ctr"].mean()
print(f"3. Average CTR for visible top-10 pages: {page1_ctr:.2%}")

1. High-traffic stale pages (>=180 days update age, >=500 impressions): 17
2. Declining pages count: 16,262 (54.2%)
3. Average CTR for visible top-10 pages: 33.89%


## 4. Careful words: what I can and can't claim

*   **What I CAN claim**:
    *   "The model provides **decision-support** by prioritizing pages for review based on historical observations of decline and freshness."
    *   "We **observe a directional association** between a page's freshness metrics (days since last update) and its performance trend."
    *   "The model offers a **prioritized ranking** of pages to optimize finite editorial capacity."
*   **What I CANNOT claim**:
    *   "We **cannot prove** that updating a page will causally increase its traffic (this would require randomized controlled trials)."
    *   "We **are not predicting Google's search algorithm** or ranking changes; we are only summarizing and modeling historical metrics."
    *   "The model does not guarantee recovery or traffic growth."

In [4]:
# Verify association between days since last update and trend direction
median_update_days = df.groupby("trend_direction")["days_since_last_update"].median()
print("Median days since last update by trend direction:")
print(median_update_days.round(1).to_string())

Median days since last update by trend direction:
trend_direction
down      20.0
flat      20.0
new       20.0
stable    22.0
up        22.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.